# Knowledge-Aware Evaluation for Legal Document Summarization — Australian-Data

This notebook runs the knowledge-aware evaluation pipeline on the Australian legal case corpus (59 documents across Murder, Robbery, Land Dispute, and Corruption categories). Intent phrases for the Australian-Data are extracted automatically by a JointBERT model trained on Indian-Data (transfer learning).

We extend the intent-based metric of Mullick et al. (2022) along two axes:

1. **Semantic Intent Metric (SIM)** — replaces the original exact-substring matching with a windowed cosine-similarity match over sentence-tuned MPNet embeddings.
2. **LLM-as-Judge** — adds a reference-free LLM-based evaluator that scores how well a summary preserves the legal intent of the source document.

**Companion notebook:** `Knowledge_Aware_Legal_Eval_Indian.ipynb` runs the same pipeline on the Indian-Data corpus.

**Runtime:** GPU recommended (T4 is sufficient). Full run takes ~30–35 minutes.

## 1. Setup


In [ ]:
!pip install -q transformers sentence-transformers torch scipy scikit-learn pandas numpy tqdm
!pip install -q anthropic openai
!pip install -q nltk rouge-score sacrebleu
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('Setup done.')

## 2. Configuration


In [ ]:
# Configuration
# LLM-Judge backend: 'anthropic', 'openai', 'llama_local', or 'skip'
LLM_JUDGE_MODE = 'llama_local'

# API keys (only needed for 'anthropic' or 'openai' backends)
ANTHROPIC_API_KEY = ''
OPENAI_API_KEY = ''

# Initial threshold for semantic intent matching; tuned automatically by the sweep cell.
SIM_THRESHOLD = 0.55

# Documents per run. Set to None to evaluate the full Australian-Data (59 docs).
MAX_DOCS = None

# Summary length as a fraction of the source document.
SUMMARY_RATIO = 0.3

import os
if ANTHROPIC_API_KEY:
    os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'LLM-Judge mode: {LLM_JUDGE_MODE}')

## 3. Dataset — Australian-Data

59 Australian legal case reports across four categories: Murder (15), Robbery (15), Corruption (15), Land Dispute (14). Sourced from the UCI Legal Case Reports dataset and annotated with intent phrases via JointBERT transfer learning from Indian-Data.

Place `legal_dataset.tar.gz` in the working directory before running the cell below.

In [ ]:
import os, tarfile, zipfile
from pathlib import Path

BASE = Path('/content') if os.path.exists('/content') else Path('.')

tarball = BASE / 'legal_dataset.tar.gz'
if tarball.exists() and not (BASE / 'dataset').exists():
    print(f'Extracting {tarball}...')
    with tarfile.open(tarball, 'r:gz') as t:
        t.extractall(BASE)
    print('Done.')

zipball = BASE / 'dataset.zip'
if zipball.exists() and not (BASE / 'dataset').exists():
    print(f'Extracting {zipball}...')
    with zipfile.ZipFile(zipball, 'r') as z:
        z.extractall(BASE)
    print('Done.')

DATASET_ROOT = BASE / 'dataset'
print(f'Dataset root: {DATASET_ROOT}  exists={DATASET_ROOT.exists()}')
if DATASET_ROOT.exists():
    aus_root = DATASET_ROOT / 'australian_data'
    if aus_root.exists():
        print('\nAustralian-Data:')
        for sub in aus_root.iterdir():
            if sub.is_dir():
                print(f'  {sub.name}/  ({len(list(sub.iterdir()))} files)')
            else:
                print(f'  {sub.name}')

In [ ]:
import csv
import re
from pathlib import Path
from collections import Counter

def parse_phrase_file(phrase_file: Path):
    """Parse intent phrase file. Some entries include 4 trailing positional integers."""
    phrases = []
    if not phrase_file.exists():
        return phrases
    with open(phrase_file, encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            m = re.match(r'^(.*?)\s+\d+\s+\d+\s+\d+\s+\d+\s*$', line)
            phrase = m.group(1).strip() if m else line
            if phrase:
                phrases.append(phrase)
    return phrases

def load_australian_data(root: Path):
    text_dir = root / 'aus_text'
    phrase_dir = root / 'aus_phrases'
    cat_map = {'dispute': 'Land Dispute', 'corruption': 'Corruption',
               'robbery': 'Robbery', 'murder': 'Murder'}
    docs = []
    for txt_file in sorted(text_dir.glob('*.txt')):
        phrase_file = phrase_dir / txt_file.name
        text = txt_file.read_text(encoding='utf-8', errors='ignore')
        phrases = parse_phrase_file(phrase_file)
        if not phrases or len(phrases[0].split()) > 50:
            phrases = [l.strip() for l in phrase_file.read_text(encoding='utf-8', errors='ignore').splitlines() if l.strip()]
        category = 'Unknown'
        for key, label in cat_map.items():
            if key in txt_file.stem.lower():
                category = label
                break
        if phrases:
            docs.append({
                'doc_id': f'AUS_{txt_file.stem}',
                'category': category,
                'text': text,
                'intent_phrases': phrases,
                'source': 'australian',
            })
    return docs

documents = []
if (DATASET_ROOT / 'australian_data').exists():
    documents = load_australian_data(DATASET_ROOT / 'australian_data')
    print(f'Loaded {len(documents)} Australian documents')
    print(f'Categories: {dict(Counter(d["category"] for d in documents))}')

if MAX_DOCS:
    documents = documents[:MAX_DOCS]

print(f'\nUsing {len(documents)} documents.')
if documents:
    sample = documents[0]
    print(f'Sample: {sample["doc_id"]} ({sample["category"]})')
    print(f'  Text length: {len(sample["text"].split())} words')
    print(f'  Intent phrases ({len(sample["intent_phrases"])}): {sample["intent_phrases"][:3]}...')

## 4. Summarization

We generate extractive summaries using a BERT-based clustering approach (Miller, 2019): sentences are encoded with `all-MiniLM-L6-v2`, clustered with k-means, and the sentence closest to each cluster centroid is selected. The number of clusters is set so the output is approximately `SUMMARY_RATIO` of the input length.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from nltk.tokenize import sent_tokenize
from tqdm.auto import tqdm
import numpy as np

print('Loading sentence encoder for summarization...')
summarizer_encoder = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)

def bert_extractive_summary(text: str, ratio: float = 0.3) -> str:
    sents = sent_tokenize(text)
    if len(sents) <= 2:
        return text
    n_summary = max(1, int(len(sents) * ratio))
    embeddings = summarizer_encoder.encode(sents, show_progress_bar=False)
    n_clusters = min(n_summary, len(sents))
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10).fit(embeddings)
    closest_indices = []
    for i in range(n_clusters):
        cluster_sents = np.where(km.labels_ == i)[0]
        if len(cluster_sents) == 0:
            continue
        cluster_embs = embeddings[cluster_sents]
        center = km.cluster_centers_[i]
        dists = np.linalg.norm(cluster_embs - center, axis=1)
        closest_indices.append(cluster_sents[np.argmin(dists)])
    closest_indices = sorted(set(closest_indices))
    return ' '.join([sents[i] for i in closest_indices])

print(f'Generating summaries (ratio={SUMMARY_RATIO})...')
for doc in tqdm(documents):
    doc['summary'] = bert_extractive_summary(doc['text'], ratio=SUMMARY_RATIO)

print(f'\nExample summary from {documents[0]["doc_id"]}:')
print(f'Summary length: {len(documents[0]["summary"].split())} words')

## 5. Original Intent Metric (Mullick et al., 2022)

Binary similarity $s_{ij}$ between intent phrase $P_i$ and summary sentence $O_j$:

$$s_{ij} = \begin{cases} 1 & \text{if } P_i \text{ is a substring of } O_j \\ 0 & \text{otherwise} \end{cases}$$

Reproduced as the baseline against which our extension is compared.

In [ ]:
def original_intent_metric(intent_phrases, summary_sentences):
    M, N = len(intent_phrases), len(summary_sentences)
    if M == 0 or N == 0:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    s = np.zeros((M, N), dtype=int)
    for i, p in enumerate(intent_phrases):
        for j, sent in enumerate(summary_sentences):
            if p.lower() in sent.lower():
                s[i, j] = 1
    P = (s.sum(axis=0) > 0).sum() / N
    R = (s.sum(axis=1) > 0).sum() / M
    F1 = 2*P*R / (P+R) if (P+R) > 0 else 0.0
    return {'precision': float(P), 'recall': float(R), 'f1': float(F1)}

for doc in documents:
    sents = sent_tokenize(doc['summary'])
    doc['original_intent'] = original_intent_metric(doc['intent_phrases'], sents)

avg_orig_f1 = np.mean([d['original_intent']['f1'] for d in documents])
print(f'Average Original Intent Metric F1: {avg_orig_f1:.4f}')

## 6. Semantic Intent Metric (SIM)

We extend the original Intent Metric in two ways:

**(1) Embedding-based similarity.** We replace the binary substring match with cosine similarity over sentence embeddings. A pair $(P_i, O_j)$ counts as a match when their similarity exceeds a threshold $\tau$:

$$s_{ij} = \mathbb{1}\left[\cos(\mathbf{e}(P_i), \mathbf{e}(O_j)) \ge \tau\right]$$

**(2) Windowed matching.** Comparing a short intent phrase against an entire summary sentence dilutes the similarity score because much of the sentence is unrelated context. We instead slide a window of length approximately equal to the phrase length across each sentence and take the maximum similarity over those windows.

We use `all-mpnet-base-v2` as the encoder.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch

print('Loading SIM encoder...')
SIM_ENCODER_NAME = 'all-mpnet-base-v2'
sim_encoder = SentenceTransformer(SIM_ENCODER_NAME, device=DEVICE)

def encode_sim(texts, batch_size=64):
    if isinstance(texts, str):
        texts = [texts]
    if not texts:
        return np.zeros((0, 768))
    return sim_encoder.encode(
        texts, batch_size=batch_size,
        convert_to_numpy=True, normalize_embeddings=True,
        show_progress_bar=False,
    )

def make_windows(sentence, window_words):
    """Sliding word-windows with 50% overlap."""
    words = sentence.split()
    if len(words) <= window_words:
        return [sentence]
    step = max(1, window_words // 2)
    windows = []
    for s in range(0, len(words) - window_words + 1, step):
        windows.append(' '.join(words[s:s + window_words]))
    last = ' '.join(words[-window_words:])
    if last != windows[-1]:
        windows.append(last)
    return windows

def semantic_intent_metric(intent_phrases, summary_sentences, threshold=0.55):
    M, N = len(intent_phrases), len(summary_sentences)
    if M == 0 or N == 0:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'sim_matrix': None}
    phrase_embs = encode_sim(intent_phrases)
    phrase_lens = [max(2, len(p.split())) for p in intent_phrases]
    sim_matrix = np.zeros((M, N))
    for j, sent in enumerate(summary_sentences):
        unique_lens = sorted(set(phrase_lens))
        len_to_emb = {L: encode_sim(make_windows(sent, L)) for L in unique_lens}
        for i in range(M):
            wemb = len_to_emb[phrase_lens[i]]
            sims = wemb @ phrase_embs[i]
            sim_matrix[i, j] = float(sims.max()) if len(sims) else 0.0
    s = (sim_matrix >= threshold).astype(int)
    P = (s.sum(axis=0) > 0).sum() / N
    R = (s.sum(axis=1) > 0).sum() / M
    F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.0
    return {
        'precision': float(P), 'recall': float(R), 'f1': float(F1),
        'sim_matrix': sim_matrix,
    }

In [ ]:
# Threshold sweep — cache similarity matrices once and threshold them in-place.
print('Caching similarity matrices...')
cached_matrices = []
for doc in tqdm(documents):
    sents = sent_tokenize(doc['summary'])
    m = semantic_intent_metric(doc['intent_phrases'], sents, threshold=0.0)
    cached_matrices.append((doc, sents, m['sim_matrix']))

def f1_at_threshold(matrices, t):
    f1s, ps, rs = [], [], []
    for doc, sents, mat in matrices:
        if mat is None or len(doc['intent_phrases']) == 0 or len(sents) == 0:
            f1s.append(0.0); ps.append(0.0); rs.append(0.0); continue
        s = (mat >= t).astype(int)
        N, M = len(sents), len(doc['intent_phrases'])
        P = (s.sum(axis=0) > 0).sum() / N
        R = (s.sum(axis=1) > 0).sum() / M
        F1 = 2*P*R/(P+R) if (P+R) > 0 else 0.0
        ps.append(P); rs.append(R); f1s.append(F1)
    return np.mean(ps), np.mean(rs), np.mean(f1s)

print('\nThreshold sweep:')
thresholds = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
sweep_results = {}
for t in thresholds:
    P, R, F1 = f1_at_threshold(cached_matrices, t)
    sweep_results[t] = (P, R, F1)
    print(f'  τ={t:.2f}  P={P:.3f}  R={R:.3f}  F1={F1:.3f}')

candidates = {t: v[2] for t, v in sweep_results.items() if 0.45 <= t <= 0.70}
if candidates:
    best_t = max(candidates, key=candidates.get)
    print(f'\nSelected τ = {best_t:.2f}')
    SIM_THRESHOLD = best_t
    for doc, sents, mat in cached_matrices:
        if mat is None:
            doc['sim'] = {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'sim_matrix': None}
            continue
        s = (mat >= SIM_THRESHOLD).astype(int)
        N, M = len(sents), len(doc['intent_phrases'])
        P = (s.sum(axis=0) > 0).sum() / N
        R = (s.sum(axis=1) > 0).sum() / M
        F1 = 2*P*R/(P+R) if (P+R) > 0 else 0.0
        doc['sim'] = {'precision': float(P), 'recall': float(R), 'f1': float(F1), 'sim_matrix': mat}
    avg_sim_f1 = np.mean([d['sim']['f1'] for d in documents])
    print(f'SIM F1 at τ={SIM_THRESHOLD}: {avg_sim_f1:.4f}')

## 7. LLM-as-Judge

An LLM is prompted with the source document, candidate summary, and category-specific intent phrases, and asked to rate intent preservation on a 1–5 Likert scale. Three backends are supported: Anthropic API, OpenAI API, or a local Qwen2.5-1.5B-Instruct for fully reproducible zero-cost evaluation.

Prompt design follows the structured-output pattern from Liu et al. (2023, G-Eval) and Zheng et al. (2023, Judging LLM-as-a-Judge): the model returns a JSON object with a numeric score and a one-sentence justification, parsed automatically.

In [ ]:
JUDGE_PROMPT_TEMPLATE = '''You are an expert legal analyst evaluating the quality of a summary of a legal case document.

The case category is: {category}

ORIGINAL DOCUMENT:
{document}

CANDIDATE SUMMARY:
{summary}

KEY INTENT PHRASES (legally significant phrases that a good summary should preserve, semantically if not verbatim):
{intent_phrases}

Rate the candidate summary on how well it preserves the legal intent of the case (i.e. preserves these key phrases or their semantic equivalents). Use this scale:
1 = Very Poor: misses almost all legal intent
2 = Poor: misses most intent
3 = Fair: captures some intent
4 = Good: captures most intent
5 = Excellent: captures all/nearly all intent

Reply with ONLY a JSON object in this exact format:
{{"score": <integer 1-5>, "reasoning": "<one sentence>"}}
'''

import json, re

def parse_judge_response(text):
    try:
        m = re.search(r'\{[^{}]*"score"[^{}]*\}', text, re.DOTALL)
        if m:
            obj = json.loads(m.group(0))
            return int(obj.get('score', 3)), obj.get('reasoning', '')
    except Exception:
        pass
    m = re.search(r'\b([1-5])\b', text)
    return (int(m.group(1)) if m else 3), text[:200]

def truncate_text(text, max_words=2000):
    words = text.split()
    return text if len(words) <= max_words else ' '.join(words[:max_words]) + ' ...[truncated]'

def judge_anthropic(prompt):
    from anthropic import Anthropic
    client = Anthropic()
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001', max_tokens=200,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return msg.content[0].text

def judge_openai(prompt):
    from openai import OpenAI
    client = OpenAI()
    resp = client.chat.completions.create(
        model='gpt-4o-mini', max_tokens=200,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return resp.choices[0].message.content

_local_judge_pipe = None
def judge_local_llama(prompt):
    global _local_judge_pipe
    if _local_judge_pipe is None:
        from transformers import pipeline
        print('Loading local judge model (one time)...')
        _local_judge_pipe = pipeline('text-generation',
            model='Qwen/Qwen2.5-1.5B-Instruct',
            device=0 if DEVICE == 'cuda' else -1,
            torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32)
    out = _local_judge_pipe(prompt, max_new_tokens=150, do_sample=False, return_full_text=False)
    return out[0]['generated_text']

JUDGE_FN = {'anthropic': judge_anthropic, 'openai': judge_openai, 'llama_local': judge_local_llama}.get(LLM_JUDGE_MODE)

if LLM_JUDGE_MODE == 'skip' or JUDGE_FN is None:
    for doc in documents:
        doc['llm_judge'] = {'score': None, 'reasoning': 'skipped'}
    print('LLM-Judge skipped.')
else:
    print(f'Running LLM-Judge ({LLM_JUDGE_MODE})...')
    for doc in tqdm(documents):
        prompt = JUDGE_PROMPT_TEMPLATE.format(
            category=doc['category'],
            document=truncate_text(doc['text'], 2000),
            summary=doc['summary'],
            intent_phrases='\n- ' + '\n- '.join(doc['intent_phrases']),
        )
        try:
            response_text = JUDGE_FN(prompt)
            score, reasoning = parse_judge_response(response_text)
        except Exception as e:
            print(f'Judge error on {doc["doc_id"]}: {e}')
            score, reasoning = 3, 'error'
        doc['llm_judge'] = {'score': score, 'reasoning': reasoning}
    valid = [d['llm_judge']['score'] for d in documents if d['llm_judge']['score'] is not None]
    if valid:
        print(f'\nAverage LLM-Judge score: {np.mean(valid):.2f}/5')

## 8. Lexical Baseline Metrics

We compute BLEU and ROUGE-L between each generated summary and a pseudo-reference summary constructed from sentences in the source containing annotated intent phrases.

In [ ]:
from rouge_score import rouge_scorer
import sacrebleu

rouge_s = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def make_pseudo_reference(doc):
    """Pseudo-reference: source sentences containing at least one annotated intent phrase."""
    sents = sent_tokenize(doc['text'])
    selected = [s for s in sents if any(p.lower() in s.lower() for p in doc['intent_phrases'])]
    return ' '.join(selected) if selected else ' '.join(sents[:3])

for doc in documents:
    ref = make_pseudo_reference(doc)
    doc['reference'] = ref
    doc['rouge_l'] = rouge_s.score(ref, doc['summary'])['rougeL'].fmeasure
    try:
        doc['bleu'] = sacrebleu.sentence_bleu(doc['summary'], [ref]).score / 100.0
    except Exception:
        doc['bleu'] = 0.0

print(f'Avg ROUGE-L: {np.mean([d["rouge_l"] for d in documents]):.4f}')
print(f'Avg BLEU:    {np.mean([d["bleu"] for d in documents]):.4f}')

## 9. Human Evaluation Module

Optional. When `SKIP_HUMAN_EVAL = True` (default), this step is bypassed and LLM-as-Judge is used as the reference signal in correlation analysis.

In [ ]:
SKIP_HUMAN_EVAL = True
RATER_NAME = 'rater_1'
N_DOCS_TO_RATE = 20

import random, csv as _csv
random.seed(42)

if not SKIP_HUMAN_EVAL:
    rate_docs = random.sample(documents, min(N_DOCS_TO_RATE, len(documents)))
    out_path = f'human_ratings_australian_{RATER_NAME}.csv'
    print(f'Rating {len(rate_docs)} summaries. Output: {out_path}')
    print('Likert scale: 1=Very Poor, 2=Poor, 3=Fair, 4=Good, 5=Excellent\n')
    ratings = []
    for k, doc in enumerate(rate_docs, 1):
        print('=' * 80)
        print(f'[{k}/{len(rate_docs)}] {doc["doc_id"]} — Category: {doc["category"]}')
        print(f'\nIntent phrases ({len(doc["intent_phrases"])}):')
        for p in doc['intent_phrases'][:8]:
            print(f'  - {p}')
        print(f'\nSummary:\n{doc["summary"][:1500]}')
        while True:
            try:
                rel = int(input('\nRelevance (1-5): ').strip())
                if 1 <= rel <= 5: break
            except (ValueError, KeyboardInterrupt):
                pass
        while True:
            try:
                rea = int(input('Readability (1-5): ').strip())
                if 1 <= rea <= 5: break
            except (ValueError, KeyboardInterrupt):
                pass
        ratings.append({'doc_id': doc['doc_id'], 'category': doc['category'],
                        'rater': RATER_NAME, 'relevance': rel, 'readability': rea,
                        'human_score': (rel + rea) / 2})
    with open(out_path, 'w', newline='') as f:
        w = _csv.DictWriter(f, fieldnames=['doc_id','category','rater','relevance','readability','human_score'])
        w.writeheader()
        w.writerows(ratings)
    print(f'\nSaved {len(ratings)} ratings to {out_path}')
else:
    print('Human evaluation skipped (SKIP_HUMAN_EVAL = True)')

### 9.1 Aggregate Multi-Rater Annotations

When multiple `human_ratings_australian_*.csv` files are present, this cell computes per-document averaged scores and pairwise Cohen's κ between raters.

In [ ]:
import glob, pandas as pd
from sklearn.metrics import cohen_kappa_score

rating_files = sorted(glob.glob('human_ratings_australian_*.csv'))
print(f'Found {len(rating_files)} rating file(s): {rating_files}')

human_score_map = {}
if rating_files:
    all_ratings = pd.concat([pd.read_csv(f) for f in rating_files], ignore_index=True)
    print(f'Total ratings: {len(all_ratings)} from {all_ratings["rater"].nunique()} rater(s)')
    agg = all_ratings.groupby('doc_id').agg(
        relevance=('relevance', 'mean'),
        readability=('readability', 'mean'),
        human_score=('human_score', 'mean'),
        n_raters=('rater', 'nunique'),
    ).reset_index()
    print('\nPer-document averaged scores:')
    print(agg.to_string(index=False))
    human_score_map = dict(zip(agg['doc_id'], agg['human_score']))
    raters = sorted(all_ratings['rater'].unique())
    if len(raters) >= 2:
        print("\nInter-annotator agreement (Cohen's κ on Relevance):")
        for i in range(len(raters)):
            for j in range(i + 1, len(raters)):
                a = all_ratings[all_ratings.rater == raters[i]].set_index('doc_id')['relevance']
                b = all_ratings[all_ratings.rater == raters[j]].set_index('doc_id')['relevance']
                common = a.index.intersection(b.index)
                if len(common) > 1:
                    k = cohen_kappa_score(a.loc[common], b.loc[common])
                    print(f'  {raters[i]} vs {raters[j]}: κ = {k:.3f} (n={len(common)})')
    agg.to_csv('human_scores_aggregated_australian.csv', index=False)
    print('\nSaved: human_scores_aggregated_australian.csv')
else:
    print('No rating files found. Falling back to LLM-as-Judge as reference signal.')

## 10. Correlation Analysis

Spearman rank correlation between each automated metric and the reference signal (human scores when present, LLM-as-Judge otherwise).

In [ ]:
import pandas as pd
from scipy.stats import spearmanr

rows = []
for doc in documents:
    rows.append({
        'doc_id': doc['doc_id'],
        'category': doc['category'],
        'BLEU': doc['bleu'],
        'ROUGE-L': doc['rouge_l'],
        'OrigIntent_F1': doc['original_intent']['f1'],
        'SIM_F1': doc['sim']['f1'],
        'LLM_Judge': doc.get('llm_judge', {}).get('score'),
    })
df = pd.DataFrame(rows)
print('Per-document scores (first 10):')
print(df.head(10).to_string(index=False))
print('\nMetric averages:')
print(df.drop(columns=['doc_id', 'category']).mean())

In [ ]:
if human_score_map:
    df_rated = df[df['doc_id'].isin(human_score_map.keys())].copy()
    df_rated['Human_Score'] = df_rated['doc_id'].map(human_score_map)
    print(f'Human scores available for {len(df_rated)} documents.\n')
    ground_truth = df_rated['Human_Score'].values
    score_df = df_rated
    gt_label = 'Human Score'
else:
    print('Using LLM-as-Judge as reference signal.\n')
    score_df = df.dropna(subset=['LLM_Judge'])
    ground_truth = score_df['LLM_Judge'].values
    gt_label = 'LLM-as-Judge'

print(f'Spearman rank correlation with {gt_label} (n={len(score_df)}):')
print('-' * 60)
results_corr = {}
for col in ['BLEU', 'ROUGE-L', 'OrigIntent_F1', 'SIM_F1', 'LLM_Judge']:
    if col not in score_df.columns:
        continue
    if col == 'LLM_Judge' and gt_label == 'LLM-as-Judge':
        continue
    if score_df[col].std() == 0:
        print(f'  {col:20s}: N/A (zero variance)')
        continue
    rho, pval = spearmanr(score_df[col].values, ground_truth)
    print(f'  {col:20s}: ρ = {rho:+.4f}  (p = {pval:.4f})')
    results_corr[col] = {'rho': float(rho), 'p': float(pval)}

## 11. Export Results


In [ ]:
import json

df.to_csv('results_per_doc_australian.csv', index=False)

summary_results = {
    'dataset': 'australian',
    'config': {
        'sim_threshold': SIM_THRESHOLD,
        'summary_ratio': SUMMARY_RATIO,
        'n_documents': len(documents),
        'llm_judge_mode': LLM_JUDGE_MODE,
    },
    'averages': {k: float(df[k].dropna().mean()) for k in ['BLEU','ROUGE-L','OrigIntent_F1','SIM_F1','LLM_Judge']},
    'sim_threshold_sweep': {f'{t:.2f}': {'P': float(p), 'R': float(r), 'F1': float(f)} for t, (p, r, f) in sweep_results.items()},
    'spearman_correlations': results_corr,
}
with open('summary_results_australian.json', 'w') as f:
    json.dump(summary_results, f, indent=2)

print('Exported: results_per_doc_australian.csv, summary_results_australian.json')
print('\nMetric averages:')
for k, v in summary_results['averages'].items():
    print(f'  {k:20s}: {v:.4f}')

## References

Chalkidis, I., Fergadiotis, M., Malakasiotis, P., Aletras, N., & Androutsopoulos, I. (2020). LEGAL-BERT: The Muppets straight out of Law School. *Findings of EMNLP 2020*.

Liu, Y., Iter, D., Xu, Y., Wang, S., Xu, R., & Zhu, C. (2023). G-Eval: NLG Evaluation using GPT-4 with Better Human Alignment. *EMNLP 2023*.

Miller, D. (2019). Leveraging BERT for Extractive Text Summarization on Lectures. *arXiv:1906.04165*.

Mullick, A., Nandy, A., Kapadnis, M. N., Patnaik, S., Raghav, R., & Kar, R. (2022). An Evaluation Framework for Legal Document Summarization. *LREC 2022*.

Reimers, N., & Gurevych, I. (2019). Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks. *EMNLP 2019*.

Song, K., Tan, X., Qin, T., Lu, J., & Liu, T.-Y. (2020). MPNet: Masked and Permuted Pre-training for Language Understanding. *NeurIPS 2020*.

Zheng, L., Chiang, W.-L., Sheng, Y., et al. (2023). Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena. *NeurIPS 2023 Datasets and Benchmarks*.